# 🔒 Task 2: Data Anonymization & Privacy

**Objective:** Anonymize the `mobile_customers` dataset to hide personal details while preserving useful information for InsightSpark's data scientists.

### 🧠 Research: Anonymization Techniques
Before writing code, we must understand the standard techniques for anonymizing data:
1. **Data Suppression (Removal):** Completely removing columns that contain highly sensitive PII and offer no analytical value (e.g., Credit Card Numbers, Security Codes, Full Names, Exact Addresses).
2. **Pseudonymization (Hashing):** Replacing an identifier with a cryptographic hash (e.g., SHA-256). This allows us to link records across tables without exposing the raw ID.
3. **Feature Extraction:** Extracting a useful, non-identifying attribute from a PII column (e.g., extracting the `@gmail.com` domain from an email address) before deleting the original column.
4. **Data Generalization (Binning):** Converting continuous, highly specific variables into broader categories or "brackets" (e.g., changing exact `Age: 28` to `Age Group: 25-34`).

### 🎯 Our Strategy for this Dataset
*   **Drop:** `name`, `username`, `address`, `current_location`, `residence`, `birthdate`, `credit_card_number`, `credit_card_security_code`, `credit_card_expire`.
*   **Hash:** Apply SHA-256 to `customer_id` to create a safe, untraceable unique key.
*   **Extract & Drop:** Extract `email_domain` from `email`, then drop `email`.
*   **Bin (Generalize):** Convert `age` and `salary` into brackets, then drop the exact numbers.

In [47]:
# ==========================================
# 🛠️ STEP 1: LOAD & PROFILE RAW DATA
# ==========================================
import pandas as pd
import numpy as np
import hashlib
import os
import warnings
warnings.filterwarnings('ignore')

# WHAT WE ARE DOING: Loading the raw dataset from the GitHub raw URL.
# WHY: This ensures our notebook is reproducible and doesn't rely on local file paths.
url = "https://raw.githubusercontent.com/beingAnujChaudhary/cba-insightspark-data-platform/main/data/raw/mobile_customers.xlsx"
customersData = pd.read_excel(url)

# WHAT WE ARE DOING: Creating a working copy of the dataframe.
# WHY: Golden Rule of Data Engineering. NEVER modify the raw data in memory. 
# If we make a mistake, we can always restart and reload the original data.
df = customersData.copy()

In [48]:
# WHAT WE ARE DOING: Displaying initial shape and columns to identify PII.
# WHY: We must visually confirm all the sensitive columns present before we start deleting or transforming them.
print(f"📊 Raw Dataset Shape: {df.shape}")
print("\n📋 Raw Columns:")
for col in df.columns:
    print(f" - {col}")

📊 Raw Dataset Shape: (10000, 19)

📋 Raw Columns:
 - Unnamed: 0
 - customer_id
 - date_registered
 - username
 - name
 - gender
 - address
 - email
 - birthdate
 - current_location
 - residence
 - employer
 - job
 - age
 - salary
 - credit_card_provider
 - credit_card_number
 - credit_card_security_code
 - credit_card_expire


In [51]:
print(df.head(3))

   Unnamed: 0                           customer_id date_registered  \
0           0  24c9d2d0-d0d3-4a90-9a3a-e00e4aac99bd      2021-09-29   
1           1  7b2bc220-0296-4914-ba46-d6cc6a55a62a      2019-08-17   
2           2  06febdf9-07fb-4a1b-87d7-a5f97d9a5faf      2019-11-01   

       username             name gender  \
0  robertsbryan  Jonathan Snyder      M   
1       egarcia  Susan Dominguez      F   
2   turnermegan     Corey Hebert      M   

                                         address                       email  \
0  24675 Susan Valley\nNorth Dianabury, MO 02475        marcus58@hotmail.com   
1   4212 Cheryl Inlet\nPort Davidmouth, NC 54884  alexanderkathy@hotmail.com   
2      07388 Coleman Prairie\nLake Amy, IA 78695             vwood@gmail.com   

   birthdate               current_location  \
0 1978-03-11     ['78.937112', '71.260464']   
1 1970-11-29  ['-24.1692185', '100.746122']   
2 2009-04-23     ['8.019908', '-19.603269']   

                                

## 🛡️ Step 2: Executing the Anonymization Pipeline

We will now apply our strategy using a sequential pipeline. We will process the data in this specific order to ensure we extract useful features *before* we delete the sensitive columns.

In [52]:
# ==========================================
# 🗑️ STEP 2A: REMOVING DIRECT IDENTIFIERS
# ==========================================

# WHAT WE ARE DOING: Defining a list of columns that contain direct PII and have no analytical value.
# WHY: Columns like names, exact addresses, and full credit card numbers are severe privacy risks. 
# Data scientists don't need a customer's exact street address or CVV code to build predictive models.
columns_to_drop = [
    'name', 
    'username', 
    'address', 
    'residence', 
    'current_location', 
    'birthdate', # We will use 'age' instead; exact birthdate is PII
    'credit_card_number', 
    'credit_card_security_code', 
    'credit_card_expire'
]

In [53]:
# WHAT WE ARE DOING: Dropping these columns from our working dataframe.
# WHY: This immediately eliminates the most sensitive, high-risk data from the dataset.
df = df.drop(columns=columns_to_drop)

print(f"✅ Dropped {len(columns_to_drop)} columns containing direct PII.")
print(f"Remaining columns: {df.columns.tolist()}")

✅ Dropped 9 columns containing direct PII.
Remaining columns: ['Unnamed: 0', 'customer_id', 'date_registered', 'gender', 'email', 'employer', 'job', 'age', 'salary', 'credit_card_provider']


In [54]:
# ==========================================
# 🔐 STEP 2B: MASKING & FEATURE EXTRACTION
# ==========================================

# 1. HASHING CUSTOMER ID (Pseudonymization)
# WHAT WE ARE DOING: Applying a cryptographic hash (SHA-256) to the customer_id.
# WHY: InsightSpark needs a unique identifier to track user behavior over time (e.g., linking app usage). 
# Hashing converts the ID into an irreversible string of characters. This prevents anyone from guessing 
# the original ID, while still allowing data scientists to group records by user.
def hash_identifier(value):
    if pd.isna(value):
        return np.nan
    return hashlib.sha256(str(value).encode('utf-8')).hexdigest()

df['customer_id_hash'] = df['customer_id'].apply(hash_identifier)
df = df.drop(columns=['customer_id']) # Drop the original raw ID

In [55]:
# 2. EXTRACTING EMAIL DOMAIN (Feature Extraction)
# WHAT WE ARE DOING: Extracting the domain part of the email address (e.g., 'gmail.com').
# WHY: The full email is PII, but the email provider is a highly useful categorical feature 
# for data scientists to understand user demographics and tech-savviness.
df['email_domain'] = df['email'].apply(
    lambda x: str(x).split('@')[1] if pd.notna(x) and '@' in str(x) else 'unknown'
)
df = df.drop(columns=['email']) # Drop the full email address

print("✅ Pseudonymized customer_id and extracted email domains.")

✅ Pseudonymized customer_id and extracted email domains.


In [56]:
# ==========================================
# 📊 STEP 2C: DATA GENERALIZATION (BINNING)
# ==========================================

# 1. BINNING AGE
# WHAT WE ARE DOING: Grouping exact ages into demographic brackets.
# WHY: An exact age combined with other data points (like job title) can uniquely identify a person 
# (this is called a linkage attack). Binning preserves the statistical distribution needed for modeling 
# while protecting individual identity.
age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)
df = df.drop(columns=['age'])

In [57]:
# 2. BINNING SALARY
# WHAT WE ARE DOING: Grouping exact salary figures into income brackets.
# WHY: Salary is highly sensitive. Binning it into ranges (e.g., '100k-150k') 
# allows data scientists to analyze purchasing power and segment customers without exposing exact earnings.
salary_bins = [0, 50000, 100000, 150000, 200000, float('inf')]
salary_labels = ['<50k', '50k-100k', '100k-150k', '150k-200k', '200k+']
df['salary_bracket'] = pd.cut(df['salary'], bins=salary_bins, labels=salary_labels, right=False)
df = df.drop(columns=['salary'])

print("✅ Generalized continuous variables (Age and Salary) into categorical brackets.")

✅ Generalized continuous variables (Age and Salary) into categorical brackets.


## 🧪 Step 3: Verification & Export

Before sharing with InsightSpark, we must verify that **zero** PII remains in the dataset. We will inspect the first few rows, format the data types, and then export the final CSV to our `deliverables/` folder.

In [58]:
# ==========================================
# 🧪 STEP 3: VERIFICATION & EXPORT
# ==========================================

# WHAT WE ARE DOING: Reordering columns for better readability and inspecting the final dataset.
# WHY: Ensures the data is clean, structured, and ready for the data science team.
final_columns = [
    'customer_id_hash', 'date_registered', 'gender', 'job', 'employer', 
    'credit_card_provider', 'email_domain', 'age_group', 'salary_bracket'
]
df_anonymized = df[final_columns].copy()

# WHAT WE ARE DOING: Converting our binned categories to 'category' data types.
# WHY: This saves memory and explicitly signals to data scientists that these are discrete groups, 
# not continuous numbers.
df_anonymized['age_group'] = df_anonymized['age_group'].astype('category')
df_anonymized['salary_bracket'] = df_anonymized['salary_bracket'].astype('category')

In [59]:
# 1. VISUAL VERIFICATION
print("🔍 Visual Verification (First 5 Rows):")
display(df_anonymized.head())

🔍 Visual Verification (First 5 Rows):


,customer_id_hash,date_registered,gender,job,employer,credit_card_provider,email_domain,age_group,salary_bracket
0,cf96b84be95602a01d470fef6c5775ae9b187421e24906...,2021-09-29,M,Chief Technology Officer,"Byrd, Welch and Holt",VISA 19 digit,hotmail.com,45-54,50k-100k
1,389c7185edc98f251ac1235788b66a03b1e394dddffd24...,2019-08-17,F,Data scientist,Hurst PLC,Discover,hotmail.com,35-44,50k-100k
2,7a9394657a1fcbe2a525e5f8f3add3329bb0022e39eb19...,2019-11-01,M,Chief Operating Officer,"Mora, Caldwell and Guerrero",VISA 16 digit,gmail.com,45-54,200k+
3,511d9da9211917b4c3f23ed14854d350edcbfe3a4b48f7...,2021-12-31,F,Counselling psychologist,Patel PLC,VISA 16 digit,gmail.com,25-34,100k-150k
4,9535dcb56772ba94bea4e23b547c522ae4c667022c07e9...,2020-08-09,F,Mining engineer,Smith-Mejia,JCB 16 digit,hotmail.com,55-64,100k-150k


In [60]:
# 2. EXPORT TO DELIVERABLES
# WHAT WE ARE DOING: Saving the anonymized dataframe to a CSV file.
# WHY: This is the final deliverable requested by the stakeholder for InsightSpark.
os.makedirs('deliverables', exist_ok=True)
output_path = 'deliverables/task2_anonymized_customers.csv'
df_anonymized.to_csv(output_path, index=False)

print(f"\n🎉 SUCCESS! Anonymized dataset exported to: {output_path}")
print("⚠️  REMINDER: Ensure 'data/raw/' is in your .gitignore so raw PII is never pushed to GitHub!")


🎉 SUCCESS! Anonymized dataset exported to: deliverables/task2_anonymized_customers.csv
⚠️  REMINDER: Ensure 'data/raw/' is in your .gitignore so raw PII is never pushed to GitHub!
